# Auction-Based Road Allocation — Interactive Simulation

**Research context.**  
Urban roads are a scarce shared resource.  When too many vehicles compete for the same segment at the same time,
congestion arises.  This notebook implements and compares *online auction mechanisms* for allocating road capacity
in a **time-expanded road network**, where each node is a (physical intersection, time-slot) pair.

**How it works.**  
Vehicles arrive sequentially, each carrying:
- an origin and destination,
- a preferred departure (or arrival) time,
- a maximum willingness to pay (`reserve`), and
- an urgency weight `alpha` that blends price and travel-time in the routing cost.

The mechanism allocates **complete routes** (bundles of segment–time pairs) via
shortest-path search on the time-expanded graph.  If a vehicle's shortest-path
cost exceeds its reserve, the vehicle is rejected.  Accepted vehicles are charged
the path cost and edge prices are updated for subsequent arrivals.

---

## Compared Strategies

| Strategy | Description |
|---|---|
| **Transport-Adapted Pricing** | Exponential price update with per-edge `vmax` scaled by demand/capacity and a travel-time component.  Designed to respect capacity constraints. |
| **Online Competitive** | Exponential update with a global parameter `r` and a global capacity scalar `s_max`.  Follows the BG-style online competitive framework (Buchbinder & Naor, 2009). |
| **Zero Pricing / Free Entry** | Prices are never updated.  Serves as a baseline: all feasible vehicles are accepted at zero toll. |
| **Static Median-Occupancy Pricing** | Runs the dynamic strategy internally for `capacity/2` allocations per edge, then freezes the price.  A static approximation to the dynamic rule. |
| **Smooth Tail** | Exponential on `[0, u0]`, cubic Hermite on `(u0, 1]`.  Reaches `vmax+1` at saturation for dual feasibility, while keeping price growth smooth near the capacity limit. |

## Output Metrics

- **Social welfare** — sum of (reserve − cost) for all served vehicles.
- **Service rate** — fraction of vehicles successfully allocated.
- **Avg travel time** — mean path length (slots) for served vehicles.
- **Avg delay** — mean entry and arrival delay relative to desired time.
- **Revenue** — total tolls collected.
- **Capacity utilisation** — fraction of segment capacity used per time slot.

---
## B · Setup

Run all cells in this section **before** anything else.  
If you are on a fresh Colab runtime, start with **B1** (clone the repo), then run **B2–B4** in order.

In [ ]:
# B1 — Clone or update the repository
import os

REPO_ROOT = "/content/transportation-auction-colab"

if not os.path.isdir(REPO_ROOT):
    !git clone https://github.com/CheckIT-App/transportation-auction-colab.git {REPO_ROOT}
else:
    print("Repo already present — pulling latest updates ...")
    !git -C {REPO_ROOT} pull


In [ ]:
# B2 — Install required packages
!pip install -q -r {REPO_ROOT}/requirements.txt


In [ ]:
# B3 — Imports and path setup
import sys, os

REPO_ROOT = "/content/transportation-auction-colab"  # same as B1
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# All scripts must run from the repo root so relative paths (graph files, cache/) resolve correctly.
os.chdir(REPO_ROOT)

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from ExpandedTimeSimulation.simulation_zefat.constants import ALL_STRATEGIES
from ExpandedTimeSimulation.simulation_zefat.colab_utils import (
    load_config_from_widgets,
    preview_network,
    generate_demand_preview,
    plot_demand_distribution,
    run_experiment,
    summarize_results,
    build_summary_tables,
    plot_results,
    export_results,
    decode_reject_reason,
    preview_network_map,
    print_vehicle_summary,
)

print("Setup complete.")

In [ ]:
# B4 — Mount Google Drive (optional but recommended for persistent cache and results)
# If running locally, skip this cell.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
    print("TIP: copy har_nof.gpickle and the cache/ folder into your Drive")
    print("     and update REPO_ROOT / graph_file paths below accordingly.")
except ImportError:
    print("Not running in Colab — Drive mount skipped.")

---
## C · Configuration

Adjust the sliders and fields below, then **run cell D1** (not this cell) to extract your settings.

> **Important:** Do not re-run *this* cell (C1) after changing sliders — re-running resets all widgets to their default values. Just move on to D1.

> **Tip — minimal first run:** keep `od_count ≤ 200`, `T ≤ 20`, `runs = 1`,
> and select only 2 strategies. Expected runtime on Colab free tier: ~60–90 s.

In [ ]:
# C1 — Configuration UI

style = {"description_width": "200px"}
layout = widgets.Layout(width="480px")

# ── Basic parameters ───────────────────────────────────────────────────────
w = {
    # Demand
    "od_count":       widgets.IntSlider(value=200, min=50, max=5000, step=50,
                          description="Vehicles per run", style=style, layout=layout),
    "num_runs":       widgets.IntSlider(value=1, min=1, max=10, step=1,
                          description="Number of runs", style=style, layout=layout),
    # Time horizon
    "max_time_slots": widgets.IntSlider(value=20, min=10, max=200, step=5,
                          description="Network horizon T (slots)", style=style, layout=layout),
    "peak_slot":      widgets.IntSlider(value=10, min=1, max=100, step=1,
                          description="Peak demand slot", style=style, layout=layout),
    "peak_sigma":     widgets.FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5,
                          description="Sigma (time spread)", style=style, layout=layout),
    "time_mode":      widgets.Dropdown(
                          options=["Only Entry", "Only Arrival", "Both"],
                          value="Both",
                          description="Time field mode", style=style, layout=layout),
    # Pricing
    "vmax":           widgets.FloatSlider(value=100.0, min=10.0, max=500.0, step=10.0,
                          description="vmax (price ceiling)", style=style, layout=layout),
    # Network
    "graph_file":     widgets.Text(value="har_nof.gpickle",
                          description="Graph file (.gpickle)", style=style, layout=layout),
    "place_name":     widgets.Text(value="Har Nof, Jerusalem, Israel",
                          description="OSM place name", style=style, layout=layout),
    # Strategies
    "strategy_keys":  widgets.SelectMultiple(
                          options=ALL_STRATEGIES,
                          value=["Zero", "Transport-Adapted Pricing"],
                          description="Strategies", style=style,
                          layout=widgets.Layout(width="480px", height="120px")),
    # Reproducibility
    "base_seed":      widgets.IntText(value=2025,
                          description="Random seed", style=style, layout=layout),
    # Output
    "excel_file":     widgets.Text(value="results.xlsx",
                          description="Output Excel file", style=style, layout=layout),
}

# ── Advanced parameters (hidden in accordion) ──────────────────────────────
w_adv = {
    "r":                 widgets.IntSlider(value=30, min=5, max=100, step=1,
                             description="r  (Online Competitive base)", style=style, layout=layout),
    "vehicle_T":         widgets.IntSlider(value=100, min=20, max=300, step=10,
                             description="vehicle_T  (max vehicle horizon)", style=style, layout=layout),
    "slot_seconds":      widgets.IntSlider(value=60, min=10, max=300, step=10,
                             description="Seconds per slot", style=style, layout=layout),
    "capacity_is_hourly":widgets.Checkbox(value=True,
                             description="Capacity is hourly", style=style),
    "smooth_tail_u0":    widgets.FloatSlider(value=0.95, min=0.5, max=1.0, step=0.01,
                             description="Smooth Tail u0", style=style, layout=layout),
    "path_solver":       widgets.Dropdown(
                             options=["bidirectional_dijkstra", "dijkstra",
                                      "astar_euclidean", "astar_fallback"],
                             value="bidirectional_dijkstra",
                             description="Path solver", style=style, layout=layout),
    "arrival_percentage": widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                             description="Arrival % (when Both mode)", style=style, layout=layout),
}
w.update(w_adv)

advanced_box = widgets.Accordion(
    children=[widgets.VBox(list(w_adv.values()))],
    selected_index=None,
)
advanced_box.set_title(0, "Advanced settings")

basic_keys = [k for k in w if k not in w_adv]
ui = widgets.VBox(
    [widgets.HTML("<h4>Basic parameters</h4>")]
    + [w[k] for k in basic_keys]
    + [widgets.HTML("<br>"), advanced_box]
)
display(ui)

---
## D · Network Preparation

Loads the OSM road network from the `.gpickle` file specified above (or downloads it from OpenStreetMap on first run).
Displays a summary and map of the physical road network.

> **Caching.**  The time-expanded graph is cached automatically to `cache/expanded_net_<hash>.pkl`.
> On Colab, point `graph_file` to a path inside your mounted Drive so the cache persists between sessions.

In [ ]:
# D1 — Extract configuration from widgets
config = load_config_from_widgets(w)
print("Configuration loaded:")
for k, v in config.items():
    print(f"  {k:25s}: {v}")

In [ ]:
# D2 — Preview the road network (no time-expansion yet)
preview_network(config)

In [ ]:
# D3 — Interactive map (folium)
# folium is already installed via requirements.txt (B2), but re-installing is safe
road_map = preview_network_map(config)
display(road_map)


---
## E · Demand Generation

Generates the synthetic vehicle fleet.  Each vehicle gets:
- a random origin–destination pair from the road network,
- a desired entry time drawn from a Gaussian centred on `peak_slot`,
- an `alpha` value (price–time urgency weight) drawn from a three-band mixture,
- a `reserve` price (willingness to pay) computed from alpha and path length.

The preview table shows the first 10 vehicles; the histograms show the full fleet distribution.

In [ ]:
# E1 — Generate vehicle fleet
vehicles, preview_df = generate_demand_preview(config)
print(f"Generated {len(vehicles):,} vehicles.\n")
display(preview_df)

In [ ]:
# E2 — Distribution plots
plot_demand_distribution(vehicles)

---
## F · Simulation

Runs each selected strategy for the configured number of repetitions.
Progress is printed run-by-run.  Results are written to the Excel file
specified in the configuration.

> This cell may take **1–5 minutes** depending on `od_count`, `T`, and the number of strategies.
> Use the minimal configuration (≤ 200 vehicles, T ≤ 20, 2 strategies, 1 run) for a quick test.

In [ ]:
# F1 — Run experiment
excel_path = run_experiment(config)
print(f"\nResults saved to: {excel_path}")

---
## G · Results Summary

Loads all output sheets from the Excel file and builds per-metric comparison tables.
Green highlighting marks the best value per column.

In [ ]:
# G1 — Load result sheets
sheets = summarize_results(excel_path)

# Show a summary table of what was loaded
sheet_info = pd.DataFrame(
    [(name, len(df), len(df.columns)) for name, df in sheets.items()],
    columns=["Sheet", "Rows", "Columns"],
).set_index("Sheet")
display(sheet_info)

In [ ]:
# G2 — Per-metric comparison tables
tables = build_summary_tables(sheets)

for metric, df in tables.items():
    display(
        df.style
          .set_caption(metric)
          .highlight_max(subset=["Mean"], color="#c6efce")
          .format("{:.3f}")
    )


In [ ]:
# G3 — Rejection reason breakdown
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, COL_STRATEGY, COL_REJECT_REASON
)

veh_df = sheets.get(SHEET_VEHICLES_TABLE, pd.DataFrame())

if not veh_df.empty and COL_REJECT_REASON in veh_df.columns:
    veh_df = veh_df.copy()
    veh_df["reject_label"] = veh_df[COL_REJECT_REASON].map(decode_reject_reason)
    breakdown = (
        veh_df.groupby([COL_STRATEGY, "reject_label"])
              .size()
              .unstack(fill_value=0)
    )
    breakdown.index.name = "Strategy"
    print("\nVehicle Outcome Breakdown")
    display(breakdown)
else:
    print("vehicles_table sheet is empty or missing reject_reason column.")

---
## H · Visualisations

Renders the full diagnostic plot suite from the saved Excel file.
Plots include:

- social welfare comparison by strategy
- acceptance rate over time
- travel time vs alpha
- delay vs price band
- revenue comparison
- capacity utilisation over time
- price evolution over time per segment
- vehicles on road over time

> All plots are generated from the saved Excel file, so you can re-run
> this cell at any time without re-running the simulation.

In [ ]:
# H1 — Full diagnostic plots
plot_results(excel_path, config)

---
## I · Export

Saves each result sheet as a CSV file and writes the experiment configuration as JSON.
The Excel file is already written by the simulation step.

In [ ]:
# I1 — Export CSVs + config JSON
export_results(excel_path, config, output_dir="experiment_output")

---
## J · Developer / Advanced Tools

Low-level inspection cells.  Useful for debugging, understanding individual vehicle
allocations, tracing edge prices, or benchmarking path solvers.  These cells are
independent — run them in any order after the simulation has completed.

In [ ]:
# J1 — Inspect a single vehicle allocation
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, COL_STRATEGY, COL_SERVED,
    COL_ALPHA, COL_RESERVE, COL_PAID_FEE,
    COL_TRAVEL_TIME, COL_ENTRY_DELAY, COL_ARRIVAL_DELAY,
    COL_REJECT_REASON,
)

VEHICLE_IDX = 0        # <-- change this to inspect a different vehicle
STRATEGY    = None     # <-- set to a strategy name string, or None for the first available

veh_df = sheets.get(SHEET_VEHICLES_TABLE, pd.DataFrame())
if veh_df.empty:
    print("vehicles_table is empty — run F1 first.")
else:
    strats = veh_df[COL_STRATEGY].unique()
    chosen_strat = STRATEGY if STRATEGY in strats else strats[0]
    subset = veh_df[veh_df[COL_STRATEGY] == chosen_strat]
    row = subset.iloc[VEHICLE_IDX]
    print(f"Vehicle #{VEHICLE_IDX}  —  strategy: {chosen_strat}")
    print(f"  served          : {row.get(COL_SERVED)}")
    print(f"  reject reason   : {decode_reject_reason(row.get(COL_REJECT_REASON, 0))}")
    print(f"  alpha           : {row.get(COL_ALPHA, '—'):.3f}")
    print(f"  reserve         : {row.get(COL_RESERVE, '—'):.3f}")
    print(f"  paid fee        : {row.get(COL_PAID_FEE, '—')}")
    print(f"  travel time     : {row.get(COL_TRAVEL_TIME, '—')} slots")
    print(f"  entry delay     : {row.get(COL_ENTRY_DELAY, '—')} slots")
    print(f"  arrival delay   : {row.get(COL_ARRIVAL_DELAY, '—')} slots")

In [ ]:
# J2 — Inspect price evolution for a single edge over time
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_EDGE_TIMESLICES, COL_STRATEGY, COL_T, COL_PRICE, COL_UTIL, COL_EDGE
)

EDGE_IDX = 0    # <-- index into the list of unique edges

ts_df = sheets.get(SHEET_EDGE_TIMESLICES, pd.DataFrame())
if ts_df.empty:
    print("edge_timeslices is empty — run F1 first.")
else:
    unique_edges = ts_df[COL_EDGE].unique()
    if EDGE_IDX >= len(unique_edges):
        print(f"EDGE_IDX={EDGE_IDX} out of range. There are {len(unique_edges)} unique edges.")
    else:
        chosen_edge = unique_edges[EDGE_IDX]
        subset = ts_df[ts_df[COL_EDGE] == chosen_edge]

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for strat, grp in subset.groupby(COL_STRATEGY):
            grp_sorted = grp.sort_values(COL_T)
            axes[0].plot(grp_sorted[COL_T], grp_sorted[COL_PRICE], label=strat, marker="o", ms=3)
            axes[1].plot(grp_sorted[COL_T], grp_sorted[COL_UTIL],  label=strat, marker="o", ms=3)

        axes[0].set_title(f"Price over time — edge {EDGE_IDX}", fontsize=12)
        axes[0].set_xlabel("Time slot"); axes[0].set_ylabel("Price")
        axes[0].legend(fontsize=9)

        axes[1].set_title(f"Utilisation over time — edge {EDGE_IDX}", fontsize=12)
        axes[1].set_xlabel("Time slot"); axes[1].set_ylabel("Utilisation")
        axes[1].axhline(1.0, color="red", linestyle="--", linewidth=0.8, label="capacity")
        axes[1].legend(fontsize=9)

        plt.tight_layout()
        plt.show()
        print(f"Edge: {chosen_edge}")

In [ ]:
# J3 — Profile runtime for a minimal run
import time

PROFILE_CONFIG = {
    "od_count": 100,
    "num_runs": 1,
    "max_time_slots": 15,
    "vmax": 100.0,
    "r": 30,
    "slot_seconds": 60,
    "vehicle_T": 50,
    "peak_slot": 8,
    "peak_sigma": 3.0,
    "strategy_keys": ["Zero", "Transport-Adapted Pricing"],
    "base_seed": 42,
    "excel_file": "_profile_run.xlsx",
    "graph_file": config.get("graph_file", "har_nof.gpickle"),
    "place_name": config.get("place_name", "Har Nof, Jerusalem, Israel"),
    "capacity_is_hourly": True,
    "smooth_tail_u0": 0.95,
}

t0 = time.perf_counter()
run_experiment(PROFILE_CONFIG)
elapsed = time.perf_counter() - t0
print(f"\nProfiling complete: {elapsed:.1f}s  ({PROFILE_CONFIG['od_count']} vehicles, "
      f"T={PROFILE_CONFIG['max_time_slots']}, {len(PROFILE_CONFIG['strategy_keys'])} strategies)")

In [ ]:
#
 
J
4
 
—
 
R
o
u
t
i
n
g
 
a
l
g
o
r
i
t
h
m
 
s
p
e
e
d
 
c
o
m
p
a
r
i
s
o
n

#
 
T
i
m
e
s
 
e
a
c
h
 
p
a
t
h
-
f
i
n
d
i
n
g
 
a
l
g
o
r
i
t
h
m
 
o
n
 
a
n
 
i
d
e
n
t
i
c
a
l
 
s
m
a
l
l
 
p
r
o
b
l
e
m
 
a
n
d
 
r
e
p
o
r
t
s

#
 
r
u
n
t
i
m
e
 
a
n
d
 
n
u
m
b
e
r
 
o
f
 
v
e
h
i
c
l
e
s
 
s
e
r
v
e
d
.
 
U
s
e
 
t
h
i
s
 
t
o
 
p
i
c
k
 
t
h
e
 
f
a
s
t
e
s
t
 
s
o
l
v
e
r

#
 
f
o
r
 
y
o
u
r
 
n
e
t
w
o
r
k
 
s
i
z
e
.

i
m
p
o
r
t
 
t
i
m
e
,
 
c
o
p
y


f
r
o
m
 
E
x
p
a
n
d
e
d
T
i
m
e
S
i
m
u
l
a
t
i
o
n
.
s
i
m
u
l
a
t
i
o
n
_
z
e
f
a
t
.
e
x
p
e
r
i
m
e
n
t
s
.
b
a
t
c
h
_
r
u
n
 
i
m
p
o
r
t
 
_
l
o
a
d
_
o
r
_
b
u
i
l
d
_
g
r
a
p
h
_
a
n
d
_
x
y

f
r
o
m
 
E
x
p
a
n
d
e
d
T
i
m
e
S
i
m
u
l
a
t
i
o
n
.
s
i
m
u
l
a
t
i
o
n
_
z
e
f
a
t
.
n
e
t
w
o
r
k
 
i
m
p
o
r
t
 
T
i
m
e
E
x
p
a
n
d
e
d
R
o
a
d
N
e
t
w
o
r
k

f
r
o
m
 
E
x
p
a
n
d
e
d
T
i
m
e
S
i
m
u
l
a
t
i
o
n
.
s
i
m
u
l
a
t
i
o
n
_
z
e
f
a
t
.
a
u
c
t
i
o
n
_
s
i
m
u
l
a
t
o
r
 
i
m
p
o
r
t
 
A
u
c
t
i
o
n
S
i
m
u
l
a
t
o
r

f
r
o
m
 
E
x
p
a
n
d
e
d
T
i
m
e
S
i
m
u
l
a
t
i
o
n
.
s
i
m
u
l
a
t
i
o
n
_
z
e
f
a
t
.
s
t
r
a
t
e
g
y
_
f
a
c
t
o
r
y
 
i
m
p
o
r
t
 
m
a
k
e
_
s
t
r
a
t
e
g
y

f
r
o
m
 
E
x
p
a
n
d
e
d
T
i
m
e
S
i
m
u
l
a
t
i
o
n
.
s
i
m
u
l
a
t
i
o
n
_
z
e
f
a
t
.
e
x
p
e
r
i
m
e
n
t
s
.
v
e
h
i
c
l
e
_
g
e
n
e
r
a
t
i
o
n
 
i
m
p
o
r
t
 
(

 
 
 
 
m
i
x
e
d
_
a
l
p
h
a
_
s
a
m
p
l
e
r
,
 
a
s
s
i
g
n
_
p
e
a
k
_
d
e
s
i
r
e
d
_
e
n
t
r
y
,
 
P
e
a
k
S
c
h
e
d
u
l
e

)


S
O
L
V
E
R
S
 
=
 
[
"
b
i
d
i
r
e
c
t
i
o
n
a
l
_
d
i
j
k
s
t
r
a
"
,
 
"
d
i
j
k
s
t
r
a
"
,
 
"
a
s
t
a
r
_
e
u
c
l
i
d
e
a
n
"
,
 
"
a
s
t
a
r
_
f
f
l
b
"
]

N
_
V
E
H
I
C
L
E
S
 
=
 
1
0
0

T
 
=
 
1
5


g
r
a
p
h
_
f
i
l
e
 
=
 
c
o
n
f
i
g
.
g
e
t
(
"
g
r
a
p
h
_
f
i
l
e
"
,
 
"
h
a
r
_
n
o
f
.
g
p
i
c
k
l
e
"
)

n
o
d
e
_
x
y
_
f
i
l
e
 
=
 
g
r
a
p
h
_
f
i
l
e
.
r
e
p
l
a
c
e
(
"
.
g
p
i
c
k
l
e
"
,
 
"
_
n
o
d
e
_
x
y
_
m
e
t
e
r
s
.
p
k
l
"
)


l
o
a
d
e
r
,
 
n
o
d
e
_
x
y
_
f
i
l
e
 
=
 
_
l
o
a
d
_
o
r
_
b
u
i
l
d
_
g
r
a
p
h
_
a
n
d
_
x
y
(

 
 
 
 
p
l
a
c
e
_
n
a
m
e
=
c
o
n
f
i
g
.
g
e
t
(
"
p
l
a
c
e
_
n
a
m
e
"
,
 
"
H
a
r
 
N
o
f
,
 
J
e
r
u
s
a
l
e
m
,
 
I
s
r
a
e
l
"
)
,

 
 
 
 
g
r
a
p
h
_
f
i
l
e
=
g
r
a
p
h
_
f
i
l
e
,

 
 
 
 
n
o
d
e
_
x
y
_
f
i
l
e
=
n
o
d
e
_
x
y
_
f
i
l
e
,

 
 
 
 
o
d
_
c
o
u
n
t
=
N
_
V
E
H
I
C
L
E
S
,

)

l
o
a
d
e
r
.
g
e
n
e
r
a
t
e
_
o
d
_
p
a
i
r
s
(
)

b
a
s
e
_
e
d
g
e
s
 
=
 
l
o
a
d
e
r
.
c
o
n
v
e
r
t
_
t
o
_
b
a
s
e
_
e
d
g
e
s
(
)


a
l
p
h
a
_
m
i
x
 
=
 
m
i
x
e
d
_
a
l
p
h
a
_
s
a
m
p
l
e
r
(
[
(
0
.
4
,
(
0
,
0
.
3
)
)
,
(
0
.
4
,
(
0
.
3
,
0
.
7
)
)
,
(
0
.
2
,
(
0
.
7
,
1
.
0
)
)
]
)

v
e
h
i
c
l
e
s
 
=
 
l
o
a
d
e
r
.
g
e
n
e
r
a
t
e
_
v
e
h
i
c
l
e
s
(
a
l
p
h
a
=
a
l
p
h
a
_
m
i
x
)

a
s
s
i
g
n
_
p
e
a
k
_
d
e
s
i
r
e
d
_
e
n
t
r
y
(

 
 
 
 
v
e
h
i
c
l
e
s
,

 
 
 
 
s
c
h
e
d
u
l
e
=
P
e
a
k
S
c
h
e
d
u
l
e
(
p
e
a
k
_
s
l
o
t
=
8
,
 
s
i
g
m
a
=
3
.
0
,
 
h
o
r
i
z
o
n
_
T
=
5
0
)
,

 
 
 
 
w
r
i
t
e
_
a
r
r
i
v
a
l
=
T
r
u
e
,

)


r
e
s
u
l
t
s
 
=
 
[
]

f
o
r
 
s
o
l
v
e
r
 
i
n
 
S
O
L
V
E
R
S
:

 
 
 
 
n
e
t
 
=
 
T
i
m
e
E
x
p
a
n
d
e
d
R
o
a
d
N
e
t
w
o
r
k
(

 
 
 
 
 
 
 
 
b
a
s
e
_
e
d
g
e
s
,
 
m
a
x
_
t
i
m
e
_
s
l
o
t
s
=
T
,
 
v
m
a
x
=
1
0
0
,
 
r
=
3
0
,

 
 
 
 
 
 
 
 
p
r
i
c
i
n
g
_
s
t
r
a
t
e
g
y
=
m
a
k
e
_
s
t
r
a
t
e
g
y
(
"
Z
e
r
o
"
)
,

 
 
 
 
 
 
 
 
n
o
d
e
_
x
y
_
f
i
l
e
=
n
o
d
e
_
x
y
_
f
i
l
e
,

 
 
 
 
)

 
 
 
 
s
i
m
 
=
 
A
u
c
t
i
o
n
S
i
m
u
l
a
t
o
r
(
n
e
t
,
 
c
o
p
y
.
d
e
e
p
c
o
p
y
(
v
e
h
i
c
l
e
s
)
,
 
p
a
t
h
_
s
o
l
v
e
r
=
s
o
l
v
e
r
)

 
 
 
 
t
0
 
=
 
t
i
m
e
.
p
e
r
f
_
c
o
u
n
t
e
r
(
)

 
 
 
 
s
i
m
.
r
u
n
(
)

 
 
 
 
e
l
a
p
s
e
d
 
=
 
t
i
m
e
.
p
e
r
f
_
c
o
u
n
t
e
r
(
)
 
-
 
t
0

 
 
 
 
s
e
r
v
e
d
 
=
 
s
u
m
(
1
 
f
o
r
 
v
 
i
n
 
s
i
m
.
v
e
h
i
c
l
e
s
 
i
f
 
v
.
g
e
t
(
"
s
e
r
v
e
d
"
)
)

 
 
 
 
r
e
s
u
l
t
s
.
a
p
p
e
n
d
(
{
"
s
o
l
v
e
r
"
:
 
s
o
l
v
e
r
,
 
"
t
i
m
e
_
s
"
:
 
r
o
u
n
d
(
e
l
a
p
s
e
d
,
 
2
)
,
 
"
s
e
r
v
e
d
"
:
 
s
e
r
v
e
d
}
)

 
 
 
 
p
r
i
n
t
(
f
"
 
 
{
s
o
l
v
e
r
:
3
0
s
}
 
{
e
l
a
p
s
e
d
:
.
2
f
}
s
 
 
 
s
e
r
v
e
d
=
{
s
e
r
v
e
d
}
/
{
N
_
V
E
H
I
C
L
E
S
}
"
)


d
i
s
p
l
a
y
(
p
d
.
D
a
t
a
F
r
a
m
e
(
r
e
s
u
l
t
s
)
.
s
e
t
_
i
n
d
e
x
(
"
s
o
l
v
e
r
"
)
)